# 02 Exploratory Data Analysis
Here we explore the data (make a note of that in the report) for things like skews, distribution of mass, years, locations, fell vs found, etc. What are the most common meteorite classes? most common spots for them to fall? We want to take a look at what the basic numbers tell us, before fanciful clustering and more later.


In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# getting our cleaned data from the processed data folder
df = pd.read_csv("../data/processed/meteorite_landings_clean.csv")

# here we can see how many rows/cols we have and the datatyles of each column
print (df.shape)
print(df.dtypes)

# now we choose some columns we want to see the skew of:
skew_cols = ["mass (g)", "reclat", "reclong"]
# My idea here is that we can see skew of location and mass, mass being obvious, but 
# latitude and longitudinal skew could be neat, if it exists
skew_table = df[skew_cols].skew(numeric_only=True).sort_values(key=np.abs, ascending=False)
skew_table
# skew table gave us:
# mass (g)    77.020563
# reclat       0.147269
# reclong      0.138476
# Which tells us that while our meteorite mass is absurdly skewed, latitude and longitude is not.
# I think though, that I may have misunderstood what skew IS. 

# Now we log transform mass due to its absurd size.
df["log_mass"] = np.log1p(df["mass (g)"])

# check skew again
print("Skew before:", df["mass (g)"].skew())
print("Skew after :", df["log_mass"].skew())

# We also want to make fell vs found binary for clustering later.
df["fall_binary"] = df["fall"].map({"Found": 0, "Fell": 1})


(45716, 10)
name               str
id               int64
nametype           str
recclass           str
mass (g)       float64
fall               str
year           float64
reclat         float64
reclong        float64
GeoLocation        str
dtype: object
Skew before: 76.91011731918955
Skew after : 0.9072237919413435


In [11]:
import pandas as pd
import geopandas as gpd
import numpy as np

# Next, lets do some grouping! using geolocation, we split each meteorite into which continent it landed on.

df["reclong_norm"] = ((df["reclong"] + 180) % 360) - 180  # normalize to [-180, 180]
lat_ok = df["reclat"].between(-90, 90)
lon_ok = df["reclong_norm"].between(-180, 180)
not_missing = df["reclat"].notna() & df["reclong_norm"].notna()
zero_zero = (df["reclat"] == 0) & (df["reclong_norm"] == 0)  # treat as placeholder

valid = not_missing & lat_ok & lon_ok & ~zero_zero
print("Valid geolocation rows:", int(valid.sum()))
print("Invalid/unknown geolocation rows:", int((~valid).sum()))

# get valid geolocations

# create a GeoDataFrame with valid geolocations
points = gpd.GeoDataFrame(
    df.loc[valid].copy(),
    geometry=gpd.points_from_xy(df.loc[valid, "reclong"], df.loc[valid, "reclat"]),
    crs="EPSG:4326" # converts it to coords
) 

# This is the world polygons dataset we downloaded from natural earth, lets us bound points to land boxes
# and determine country and continent!
world = gpd.read_file("../data/external/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp").to_crs("EPSG:4326")
land = world[["ADMIN", "CONTINENT", "geometry"]].rename(
    columns={"ADMIN": "country_land", "CONTINENT": "continent_land"}
)

# this is where geo pandas checks which points meteorites are in.
joined = gpd.sjoin(points, land, how="left", predicate="intersects")
joined = joined[~joined.index.duplicated(keep="first")]  # border-safe

#  This is for the invalid coords 
df["country"] = "Unknown"
df["continent"] = "Unknown"

# valid coords but no land, so ocean
df.loc[valid, "country"] = "Open Ocean"   
df.loc[valid, "continent"] = "Open Ocean"

# and now we fill the valid coords if they have a country_land or continent_land!
df.loc[joined.index, "country"] = joined["country_land"].fillna("Open Ocean")
df.loc[joined.index, "continent"] = joined["continent_land"].fillna("Open Ocean")

df[["country", "continent"]].head()

# before we save, also get the distance from the equator in kilometers.
df["dist_equator_km"] = df["reclat"].abs() * 111.32


# Now we save our new and improved preprocessed dataset!
df.to_csv("../data/processed/meteorite_landings_processed.csv", index=False)

Valid geolocation rows: 32187
Invalid/unknown geolocation rows: 13529
